Random Forest - General Building Process  

1) create a bootstrap data set (random sampling with replacement)
2) create a decision tree with the bootstrap dataset but only use a random subset of variables at each step (can optimize, example: 2)
3) repeat and build 100s-1000s of trees, variety of decision trees
4) run new data through all the trees
5) get aggregate of all predicted values to make a final prediction

Bootstrapping the data plus using the aggregate to make a decision is called "Bagging"

Out-of-Bag Dataset --> the entries that did not make it into bootstrap data, they go through all of the trees, compare predicted result vs actual 

Can measure how accurate a random forest is by the proportion of out-of-bag samples that were correctly predicted by the random forest
out-of-bag error --> the proportion of sampels that were incorrectly predicted

Can use out-of-bag error to test different settings used in building a random forest --> may have originally tested just 2 variables at once when creating a regression tree step, can change that to 3 or more


Random Forest - Missing Data & Clustering

2 types of missing data
1) missing data in the original dataset used to create a random forest
    - initial guess at what missing values might be, using data (categories, means) from other samples with same y-value
    - run data with filled in data values down regression trees
    - build a proximity matrix (do rows end in the same leaf with each tree?) to measure similarity among samples/rows
    - divide proximity values numbers by number of trees
    - proximity values are used as weights to better guess what missing data values might be
    - repeat the entire process of building a random forest, running data through the trees, revising guesses with new proximities...this is done ~6-7 times until the missing values converge (no longer change with each recalculation)

    * proximity matrices can be used to make heat maps and visualize clustering with MultiDimensional Scaling (MDS) -- similar to PCA, where PCA is a type of analysis and MDS is more of an umbrella term that can include PCA

2) missing data in a new sample that we want to predict (unkown x's and y)
    - create copies of the data where y is assigned a value
    - go through the iterative process described above of guessing x-values through all the trees in a random forest
    

In [35]:
#Import libraries
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor


In [36]:
#CREATE RANDOM FOREST MODELS FOR ESTIMATING TOTAL OYSTER DENSITY USING CODED PERCENT COVER (replaced coverage codes for 2023-2024 data with 0-4)

#Load data
df = pd.read_csv('2023_2025OSdata.csv')

#drop sites where oysters were not counted
df = df.dropna(subset=['total'])
row_count = len(df)
print(row_count)

269


In [37]:
#extract2 removed some strange %cover:density numbers at Crab Hole in 2024, and extreme outlier at Swan Island 2024
CH24 = df[(df['OS_ID'] == 5) & (df['Year'] == 2024)].index
#outlier = df[(df['total'] == 6776)].index
extract = df.drop(CH24)
#extract = df.drop(outlier)

# Select features
#can't use material as a parameter because different between excavated vs observational sites. 
# Also, tried including material as a parameter and it had very low significance in the model
#full -->['OS_ID', 'Material_Age', 'Oyster.Cover', 'Mussel_Cover', 'Sedimentation', 'Boring_Sponge', 'Sample_Depth', 'OS.Depth', 'Relief', 'B.DO', 'B.Sal']
X = extract[['Year', 'OS_ID', 'Material_Age', 'CoverCode', 'Sedimentation', 'Boring_Sponge', 'Sample_Depth', 'Relief', 'B.DO', 'B.Sal' ]]

#small epsilon value to avoid log(0)
epsilon = 1e-9

# Define target variables
ynorm_total = extract['total']
ysqrt_total = np.sqrt(extract['total']) #apply square root transformation
ylog_total = np.log10(extract['total']+epsilon) #apply log base 10 transformation
yln_total = np.log(extract['total']+epsilon) #apply natural log (ln) transformation

# Define models and pair them with corresponding y-variables
models_total = {
    'randfor_norm_total' : (RandomForestRegressor(n_estimators=500, random_state=0), ynorm_total),
    'randfor_sqrt_total': (RandomForestRegressor(n_estimators=500, random_state=0), ysqrt_total),
    'randfor_log_total': (RandomForestRegressor(n_estimators=500, random_state=0), ylog_total),
    'randfor_ln_total': (RandomForestRegressor(n_estimators=500, random_state=0), yln_total),
}

# Dictionary to store results
results_total = {}
trained_models_total = {}

# Loop through each model and target variable
for model_name, (model, y) in models_total.items():
    # Split data into training and test sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=1/3, random_state=0)
    
    # Fit the model
    model.fit(X_train, y_train)
    
    # Predict
    y_pred = model.predict(X_test)
    
    # Calculate R^2 score
    r2 = r2_score(y_test, y_pred)

    # Calculate MAE -- on average the model's predictions are [MAE] units off from the actual values
    mae = mean_absolute_error(y_test, y_pred)

    # Calculate MSE -- penalizes large errors (sensitive to outliers); high value may indicate presence of significant outliers
    # models that are robust to outliers --> random forests, XGBoost
    mse = mean_squared_error(y_test, y_pred)
    
    # Calculate RMSE -- typical magnitude of error in predictions
    rmse = mse ** 0.5
    
    # Store results and trained model
    results_total[model_name] = {
        'r2_score': r2,
        'y_test': y_test,
        'y_pred': y_pred,
        'mae' : mae,
        'rmse' : rmse
    }
    trained_models_total[model_name] = model
    

# Compare results
for model_name, result in results_total.items():
    print(f"{model_name}: R^2 Score = {result['r2_score']:.4f}, MAE = {result['mae']:.4f}, RMSE = {result['rmse']:.4f}")


randfor_norm_total: R^2 Score = 0.7858, MAE = 423.9720, RMSE = 568.6084
randfor_sqrt_total: R^2 Score = 0.7922, MAE = 5.5647, RMSE = 7.1841
randfor_log_total: R^2 Score = 0.7547, MAE = 0.1702, RMSE = 0.2240
randfor_ln_total: R^2 Score = 0.7546, MAE = 0.3922, RMSE = 0.5159


In [38]:
#Square root transformation is best because highest r^2, but they all seem comprable

#Further evaluate the sqrt model, generate metrics
# with count data in random forest models, there is no explicit argument for specifying a poisson family distribution
# random forests are non-parametric so they do not assume a specific distribution of the response varialbe
X = extract[['Year', 'OS_ID', 'Material_Age', 'CoverCode', 'Sedimentation', 'Boring_Sponge', 'Sample_Depth', 'Relief', 'B.DO', 'B.Sal']]
ysqrt_total = np.sqrt(extract['total']) #apply square root transformation

# Split data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, ysqrt_total, test_size=1/3, random_state=0)

#Create and fit the model
randfor_sqrt_total = RandomForestRegressor(n_estimators=500, random_state=0)
randfor_sqrt_total.fit(X_train, y_train)

# Predict
y_pred = randfor_sqrt_total.predict(X_test)

#R-Squared -- how much of the variance is explained by the model 
r2 = randfor_sqrt_total.score(X_test, y_test)
print(f"R^2 Score: {r2}")

# Calculate MAE -- on average the model's predictions are [MAE] units off from the actual values
mae = mean_absolute_error(y_test, y_pred)
print(f"Mean Absolute Error: {mae}")

# Calculate MSE -- penalizes large errors (sensitive to outliers); high value may indicate presence of significant outliers
# models that are robust to outliers --> random forests, XGBoost
mse = mean_squared_error(y_test, y_pred)
print(f"Mean Squared Error: {mse}")

# Calculate RMSE -- typical magnitude of error in predictions
rmse = mse ** 0.5
print(f"Root Mean Squared Error: {rmse}")

#Evaluate feature importance
feature_names = X_train.columns
importances = randfor_sqrt_total.feature_importances_
feature_importance = list(zip(feature_names, importances))
feature_importance.sort(key=lambda x: x[1], reverse = True)
for feature, importance in feature_importance:
    print(f"Feature: {feature}, Importance: {importance}")



R^2 Score: 0.7921698588083559
Mean Absolute Error: 5.564681309880111
Mean Squared Error: 51.61150478896461
Root Mean Squared Error: 7.184114753326579
Feature: Material_Age, Importance: 0.27484257951770735
Feature: OS_ID, Importance: 0.27187392976558483
Feature: CoverCode, Importance: 0.20131592135859536
Feature: B.Sal, Importance: 0.08468012130710861
Feature: Sample_Depth, Importance: 0.06957914145934617
Feature: B.DO, Importance: 0.04205670495218546
Feature: Relief, Importance: 0.025187992721226026
Feature: Sedimentation, Importance: 0.010605102903882297
Feature: Boring_Sponge, Importance: 0.01008896475329837
Feature: Year, Importance: 0.009769541261065362


In [40]:
#RUN PROCESS ON LEGAL OYSTERS
#CREATE RANDOM FOREST MODELS FOR ESTIMATING LEGAL OYSTER DENSITY USING CODED PERCENT COVER (replaced coverage codes for 2023-2024 data with 0-4)
#Load data
df = pd.read_csv('2023_2025OSdata.csv')

#drop sites where oysters were not counted
df = df.dropna(subset=['total'])

#extract2 removed some strange %cover:density numbers at Crab Hole in 2024, and extreme outlier at Swan Island 2024
CH24 = df[(df['OS_ID'] == 5) & (df['Year'] == 2024)].index
#outlier = df[(df['total'] == 6776)].index
extract = df.drop(CH24)
#extract = df.drop(outlier)

# Select features
#can't use material as a parameter because different between excavated vs observational sites. 
# Also, tried including material as a parameter and it had very low significance in the model
#full -->['OS_ID', 'Material_Age', 'Oyster.Cover', 'Mussel_Cover', 'Sedimentation', 'Boring_Sponge', 'Sample_Depth', 'OS.Depth', 'Relief', 'B.DO', 'B.Sal']
X = extract[['Year', 'OS_ID', 'Material_Age', 'CoverCode', 'Sedimentation', 'Boring_Sponge', 'Sample_Depth', 'Relief', 'B.DO', 'B.Sal' ]]

#small epsilon value to avoid log(0)
epsilon = 1e-9


# Define target variables
ynorm_legal = extract['legal']
ysqrt_legal = np.sqrt(extract['legal']) #apply square root transformation
ylog_legal = np.log10(extract['legal']+epsilon) #apply log base 10 transformation
yln_legal = np.log(extract['legal']+epsilon) #apply natural log (ln) transformation

# Define models and pair them with corresponding y-variables
models_legal = {
    'randfor_norm_legal' : (RandomForestRegressor(n_estimators=500, random_state=0), ynorm_legal),
    'randfor_sqrt_legal': (RandomForestRegressor(n_estimators=500, random_state=0), ysqrt_legal),
    'randfor_log_legal': (RandomForestRegressor(n_estimators=500, random_state=0), ylog_legal),
    'randfor_ln_legal': (RandomForestRegressor(n_estimators=500, random_state=0), yln_legal),
}

# Dictionary to store results
results_legal = {}
trained_models_legal = {}

# Loop through each model and target variable
for model_name, (model, y) in models_legal.items():
    # Split data into training and test sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=1/3, random_state=0)
    
    # Fit the model
    model.fit(X_train, y_train)
    
    # Predict
    y_pred = model.predict(X_test)
    
    # Calculate R^2 score
    r2 = r2_score(y_test, y_pred)

    # Calculate MAE -- on average the model's predictions are [MAE] units off from the actual values
    mae = mean_absolute_error(y_test, y_pred)

    # Calculate MSE -- penalizes large errors (sensitive to outliers); high value may indicate presence of significant outliers
    # models that are robust to outliers --> random forests, XGBoost
    mse = mean_squared_error(y_test, y_pred)
    
    # Calculate RMSE -- typical magnitude of error in predictions
    rmse = mse ** 0.5
    
    # Store results and trained model
    results_legal[model_name] = {
        'r2_score': r2,
        'y_test': y_test,
        'y_pred': y_pred,
        'mae' : mae,
        'rmse' : rmse
    }
    trained_models_legal[model_name] = model
    

# Compare results
for model_name, result in results_legal.items():
    print(f"{model_name}: R^2 Score = {result['r2_score']:.4f}, MAE = {result['mae']:.4f}, RMSE = {result['rmse']:.4f}")



randfor_norm_legal: R^2 Score = 0.6222, MAE = 62.0996, RMSE = 94.0904
randfor_sqrt_legal: R^2 Score = 0.6752, MAE = 2.5909, RMSE = 3.5083
randfor_log_legal: R^2 Score = 0.1387, MAE = 0.6770, RMSE = 1.5984
randfor_ln_legal: R^2 Score = 0.1585, MAE = 1.5440, RMSE = 3.6378


In [41]:
#Square root transformation is best because highest r^2

#Further evaluate the sqrt model, generate metrics
# with count data in random forest models, there is no explicit argument for specifying a poisson family distribution
# random forests are non-parametric so they do not assume a specific distribution of the response varialbe
X = extract[['Year', 'OS_ID', 'Material_Age', 'CoverCode', 'Sedimentation', 'Boring_Sponge', 'Sample_Depth', 'Relief', 'B.DO', 'B.Sal']]
ysqrt_legal = np.sqrt(extract['legal']) #apply square root transformation

# Split data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, ysqrt_legal, test_size=1/3, random_state=0)

#Create and fit the model
randfor_sqrt_legal = RandomForestRegressor(n_estimators=500, random_state=0)
randfor_sqrt_legal.fit(X_train, y_train)

# Predict
y_pred = randfor_sqrt_legal.predict(X_test)

#R-Squared -- how much of the variance is explained by the model 
r2 = randfor_sqrt_legal.score(X_test, y_test)
print(f"R^2 Score: {r2}")

# Calculate MAE -- on average the model's predictions are [MAE] units off from the actual values
mae = mean_absolute_error(y_test, y_pred)
print(f"Mean Absolute Error: {mae}")

# Calculate MSE -- penalizes large errors (sensitive to outliers); high value may indicate presence of significant outliers
# models that are robust to outliers --> random forests, XGBoost
mse = mean_squared_error(y_test, y_pred)
print(f"Mean Squared Error: {mse}")

# Calculate RMSE -- typical magnitude of error in predictions
rmse = mse ** 0.5
print(f"Root Mean Squared Error: {rmse}")

#Evaluate feature importance
feature_names = X_train.columns
importances = randfor_sqrt_legal.feature_importances_
feature_importance = list(zip(feature_names, importances))
feature_importance.sort(key=lambda x: x[1], reverse = True)
for feature, importance in feature_importance:
    print(f"Feature: {feature}, Importance: {importance}")

R^2 Score: 0.6751537801906224
Mean Absolute Error: 2.590860907132152
Mean Squared Error: 12.307855163668952
Root Mean Squared Error: 3.5082552876991366
Feature: CoverCode, Importance: 0.4883842779532204
Feature: Material_Age, Importance: 0.12410135814270586
Feature: B.Sal, Importance: 0.07886448179234219
Feature: Sample_Depth, Importance: 0.06289035467421547
Feature: B.DO, Importance: 0.05162568922526523
Feature: Relief, Importance: 0.048975158230495655
Feature: Boring_Sponge, Importance: 0.048163106847940035
Feature: OS_ID, Importance: 0.046410817159331624
Feature: Sedimentation, Importance: 0.040015554467881055
Feature: Year, Importance: 0.010569201506602444


In [42]:
#RUN PROCESS ON SUBLEGAL OYSTERS
#CREATE RANDOM FOREST MODELS FOR ESTIMATING SUBLEGAL OYSTER DENSITY USING CODED PERCENT COVER (replaced coverage codes for 2023-2024 data with 0-4)
#Load data
df = pd.read_csv('2023_2025OSdata.csv')

#drop sites where oysters were not counted
df = df.dropna(subset=['total'])

#extract2 removed some strange %cover:density numbers at Crab Hole in 2024, and extreme outlier at Swan Island 2024
CH24 = df[(df['OS_ID'] == 5) & (df['Year'] == 2024)].index
#outlier = df[(df['total'] == 6776)].index
extract = df.drop(CH24)
#extract = df.drop(outlier)

# Select features
#can't use material as a parameter because different between excavated vs observational sites. 
# Also, tried including material as a parameter and it had very low significance in the model
#full -->['OS_ID', 'Material_Age', 'Oyster.Cover', 'Mussel_Cover', 'Sedimentation', 'Boring_Sponge', 'Sample_Depth', 'OS.Depth', 'Relief', 'B.DO', 'B.Sal']
X = extract[['Year', 'OS_ID', 'Material_Age', 'CoverCode', 'Sedimentation', 'Boring_Sponge', 'Sample_Depth', 'Relief', 'B.DO', 'B.Sal' ]]

#small epsilon value to avoid log(0)
epsilon = 1e-9

# Define target variables
ynorm_sublegal = extract['sublegal']
ysqrt_sublegal = np.sqrt(extract['sublegal']) #apply square root transformation
ylog_sublegal = np.log10(extract['sublegal']+epsilon) #apply log base 10 transformation
yln_sublegal = np.log(extract['sublegal']+epsilon) #apply natural log (ln) transformation

# Define models and pair them with corresponding y-variables
models_sublegal = {
    'randfor_norm_sublegal' : (RandomForestRegressor(n_estimators=500, random_state=0), ynorm_sublegal),
    'randfor_sqrt_sublegal': (RandomForestRegressor(n_estimators=500, random_state=0), ysqrt_sublegal),
    'randfor_log_sublegal': (RandomForestRegressor(n_estimators=500, random_state=0), ylog_sublegal),
    'randfor_ln_sublegal': (RandomForestRegressor(n_estimators=500, random_state=0), yln_sublegal),
}

# Dictionary to store results
results_sublegal = {}
trained_models_sublegal = {}

# Loop through each model and target variable
for model_name, (model, y) in models_sublegal.items():
    # Split data into training and test sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=1/3, random_state=0)
    
    # Fit the model
    model.fit(X_train, y_train)
    
    # Predict
    y_pred = model.predict(X_test)
    
    # Calculate R^2 score
    r2 = r2_score(y_test, y_pred)

    # Calculate MAE -- on average the model's predictions are [MAE] units off from the actual values
    mae = mean_absolute_error(y_test, y_pred)

    # Calculate MSE -- penalizes large errors (sensitive to outliers); high value may indicate presence of significant outliers
    # models that are robust to outliers --> random forests, XGBoost
    mse = mean_squared_error(y_test, y_pred)
    
    # Calculate RMSE -- typical magnitude of error in predictions
    rmse = mse ** 0.5
    
    # Store results and trained model
    results_sublegal[model_name] = {
        'r2_score': r2,
        'y_test': y_test,
        'y_pred': y_pred,
        'mae' : mae,
        'rmse' : rmse
    }
    trained_models_sublegal[model_name] = model
    

# Compare results
for model_name, result in results_sublegal.items():
    print(f"{model_name}: R^2 Score = {result['r2_score']:.4f}, MAE = {result['mae']:.4f}, RMSE = {result['rmse']:.4f}")


randfor_norm_sublegal: R^2 Score = 0.6972, MAE = 191.9566, RMSE = 277.7128
randfor_sqrt_sublegal: R^2 Score = 0.7346, MAE = 4.0089, RMSE = 5.2134
randfor_log_sublegal: R^2 Score = 0.7099, MAE = 0.1741, RMSE = 0.2322
randfor_ln_sublegal: R^2 Score = 0.7048, MAE = 0.4029, RMSE = 0.5393


In [43]:
#Square root transformation is best because highest r^2

#Further evaluate the sqrt model, generate metrics
# with count data in random forest models, there is no explicit argument for specifying a poisson family distribution
# random forests are non-parametric so they do not assume a specific distribution of the response varialbe
X = extract[['Year', 'OS_ID', 'Material_Age', 'CoverCode', 'Sedimentation', 'Boring_Sponge', 'Sample_Depth', 'Relief', 'B.DO', 'B.Sal']]
ysqrt_sublegal = np.sqrt(extract['sublegal']) #apply square root transformation

# Split data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, ysqrt_sublegal, test_size=1/3, random_state=0)

#Create and fit the model
randfor_sqrt_sublegal = RandomForestRegressor(n_estimators=500, random_state=0)
randfor_sqrt_sublegal.fit(X_train, y_train)

# Predict
y_pred = randfor_sqrt_sublegal.predict(X_test)

#R-Squared -- how much of the variance is explained by the model 
r2 = randfor_sqrt_sublegal.score(X_test, y_test)
print(f"R^2 Score: {r2}")

# Calculate MAE -- on average the model's predictions are [MAE] units off from the actual values
mae = mean_absolute_error(y_test, y_pred)
print(f"Mean Absolute Error: {mae}")

# Calculate MSE -- penalizes large errors (sensitive to outliers); high value may indicate presence of significant outliers
# models that are robust to outliers --> random forests, XGBoost
mse = mean_squared_error(y_test, y_pred)
print(f"Mean Squared Error: {mse}")

# Calculate RMSE -- typical magnitude of error in predictions
rmse = mse ** 0.5
print(f"Root Mean Squared Error: {rmse}")

#Evaluate feature importance
feature_names = X_train.columns
importances = randfor_sqrt_sublegal.feature_importances_
feature_importance = list(zip(feature_names, importances))
feature_importance.sort(key=lambda x: x[1], reverse = True)
for feature, importance in feature_importance:
    print(f"Feature: {feature}, Importance: {importance}")

R^2 Score: 0.7346227728899519
Mean Absolute Error: 4.008916985678725
Mean Squared Error: 27.179113049410365
Root Mean Squared Error: 5.213359094615521
Feature: CoverCode, Importance: 0.2966811825073346
Feature: Material_Age, Importance: 0.2932258902498297
Feature: OS_ID, Importance: 0.19152509377262386
Feature: Sample_Depth, Importance: 0.07846113291840065
Feature: B.Sal, Importance: 0.03390574339033113
Feature: B.DO, Importance: 0.03281482673276742
Feature: Relief, Importance: 0.03008146066709359
Feature: Sedimentation, Importance: 0.02108730868527522
Feature: Year, Importance: 0.014274835284640778
Feature: Boring_Sponge, Importance: 0.00794252579170326


In [44]:
#RUN PROCESS ON SPAT RECRUITS
#CREATE RANDOM FOREST MODELS FOR ESTIMATING TOTAL OYSTER DENSITY USING CODED PERCENT COVER (replaced coverage codes for 2023-2024 data with 0-4)

#Load data
df = pd.read_csv('2023_2025OSdata.csv')

#drop sites where oysters were not counted
df = df.dropna(subset=['total'])

#extract2 removed some strange %cover:density numbers at Crab Hole in 2024, and extreme outlier at Swan Island 2024
CH24 = df[(df['OS_ID'] == 5) & (df['Year'] == 2024)].index
#outlier = df[(df['total'] == 6776)].index
extract = df.drop(CH24)
#extract = df.drop(outlier)

# Select features
#can't use material as a parameter because different between excavated vs observational sites. 
# Also, tried including material as a parameter and it had very low significance in the model
#full -->['OS_ID', 'Material_Age', 'CoverCode', 'Mussel_Cover', 'Sedimentation', 'Boring_Sponge', 'Sample_Depth', 'OS.Depth', 'Relief', 'B.DO', 'B.Sal']
X = extract[['Year', 'OS_ID', 'Material_Age', 'CoverCode', 'Sedimentation', 'Boring_Sponge', 'Sample_Depth', 'Relief', 'B.DO', 'B.Sal' ]]

#small epsilon value to avoid log(0)
epsilon = 1e-9

# Define target variables
ynorm_spat = extract['spat']
ysqrt_spat = np.sqrt(extract['spat']) #apply square root transformation
ylog_spat = np.log10(extract['spat']+epsilon) #apply log base 10 transformation
yln_spat = np.log(extract['spat']+epsilon) #apply natural log (ln) transformation

# Define models and pair them with corresponding y-variables
models_spat = {
    'randfor_norm_spat' : (RandomForestRegressor(n_estimators=500, random_state=0), ynorm_spat),
    'randfor_sqrt_spat': (RandomForestRegressor(n_estimators=500, random_state=0), ysqrt_spat),
    'randfor_log_spat': (RandomForestRegressor(n_estimators=500, random_state=0), ylog_spat),
    'randfor_ln_spat': (RandomForestRegressor(n_estimators=500, random_state=0), yln_spat),
}

# Dictionary to store results
results_spat = {}
trained_models_spat = {}

# Loop through each model and target variable
for model_name, (model, y) in models_spat.items():
    # Split data into training and test sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=1/3, random_state=0)
    
    # Fit the model
    model.fit(X_train, y_train)
    
    # Predict
    y_pred = model.predict(X_test)
    
    # Calculate R^2 score
    r2 = r2_score(y_test, y_pred)

    # Calculate MAE -- on average the model's predictions are [MAE] units off from the actual values
    mae = mean_absolute_error(y_test, y_pred)

    # Calculate MSE -- penalizes large errors (sensitive to outliers); high value may indicate presence of significant outliers
    # models that are robust to outliers --> random forests, XGBoost
    mse = mean_squared_error(y_test, y_pred)
    
    # Calculate RMSE -- typical magnitude of error in predictions
    rmse = mse ** 0.5
    
    # Store results and trained model
    results_spat[model_name] = {
        'r2_score': r2,
        'y_test': y_test,
        'y_pred': y_pred,
        'mae' : mae,
        'rmse' : rmse
    }
    trained_models_spat[model_name] = model
    

# Compare results
for model_name, result in results_spat.items():
    print(f"{model_name}: R^2 Score = {result['r2_score']:.4f}, MAE = {result['mae']:.4f}, RMSE = {result['rmse']:.4f}")

randfor_norm_spat: R^2 Score = 0.8069, MAE = 247.6176, RMSE = 377.8314
randfor_sqrt_spat: R^2 Score = 0.8044, MAE = 4.6085, RMSE = 5.9154
randfor_log_spat: R^2 Score = 0.2184, MAE = 0.3629, RMSE = 1.1885
randfor_ln_spat: R^2 Score = 0.2179, MAE = 0.8347, RMSE = 2.7375


In [45]:
#Square root transformation is best because highest r^2

#Further evaluate the sqrt model, generate metrics
# with count data in random forest models, there is no explicit argument for specifying a poisson family distribution
# random forests are non-parametric so they do not assume a specific distribution of the response varialbe
X = extract[['Year', 'OS_ID', 'Material_Age', 'CoverCode', 'Sedimentation', 'Boring_Sponge', 'Sample_Depth', 'Relief', 'B.DO', 'B.Sal']]
ysqrt_spat = np.sqrt(extract['spat']) #apply square root transformation

# Split data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, ysqrt_spat, test_size=1/3, random_state=0)

#Create and fit the model
randfor_sqrt_spat = RandomForestRegressor(n_estimators=500, random_state=0)
randfor_sqrt_spat.fit(X_train, y_train)

# Predict
y_pred = randfor_sqrt_spat.predict(X_test)

#R-Squared -- how much of the variance is explained by the model 
r2 = randfor_sqrt_spat.score(X_test, y_test)
print(f"R^2 Score: {r2}")

# Calculate MAE -- on average the model's predictions are [MAE] units off from the actual values
mae = mean_absolute_error(y_test, y_pred)
print(f"Mean Absolute Error: {mae}")

# Calculate MSE -- penalizes large errors (sensitive to outliers); high value may indicate presence of significant outliers
# models that are robust to outliers --> random forests, XGBoost
mse = mean_squared_error(y_test, y_pred)
print(f"Mean Squared Error: {mse}")

# Calculate RMSE -- typical magnitude of error in predictions
rmse = mse ** 0.5
print(f"Root Mean Squared Error: {rmse}")

#Evaluate feature importance
feature_names = X_train.columns
importances = randfor_sqrt_spat.feature_importances_
feature_importance = list(zip(feature_names, importances))
feature_importance.sort(key=lambda x: x[1], reverse = True)
for feature, importance in feature_importance:
    print(f"Feature: {feature}, Importance: {importance}")

R^2 Score: 0.8044382912223378
Mean Absolute Error: 4.608547662246286
Mean Squared Error: 34.99188696175682
Root Mean Squared Error: 5.9153940664808475
Feature: OS_ID, Importance: 0.3407579653944797
Feature: Material_Age, Importance: 0.22379695659010856
Feature: CoverCode, Importance: 0.1394366847316447
Feature: B.Sal, Importance: 0.11855428695410179
Feature: Sample_Depth, Importance: 0.07650387845267174
Feature: B.DO, Importance: 0.045157183945636836
Feature: Relief, Importance: 0.03240353190669479
Feature: Year, Importance: 0.009329723218284198
Feature: Boring_Sponge, Importance: 0.009097591767898393
Feature: Sedimentation, Importance: 0.004962197038479276


In [46]:
#test and make predictions on the observational data
#import codified monitoring dataset
pred = pd.read_csv('2023_2025OSdata.csv')

pred_X = pred[['Year', 'OS_ID', 'Material_Age', 'CoverCode', 'Sedimentation', 'Boring_Sponge', 'Sample_Depth', 'Relief', 'B.DO', 'B.Sal']]

# Use the square root models to make predictions for total density, legal, and recruits

#TOTAL
randfor_sqrt_total = trained_models_total['randfor_sqrt_total']  # Retrieve the trained square root model
sqrt_pred_total = randfor_sqrt_total.predict(pred_X) #make predictions with obs_X features
sqrt_total = np.array(sqrt_pred_total) # Convert predictions to numpy array
pred['pred_total'] = sqrt_total**2 #reverse the square root transformation

#LEGAL
randfor_sqrt_legal = trained_models_legal['randfor_sqrt_legal']  # Retrieve the trained square root model
sqrt_pred_legal = randfor_sqrt_legal.predict(pred_X) #make predictions with obs_X features
sqrt_legal = np.array(sqrt_pred_legal) # Convert predictions to numpy array
pred['pred_legal'] = sqrt_legal**2 #reverse the square root transformation

#SUBLEGAL
randfor_sqrt_sublegal = trained_models_sublegal['randfor_sqrt_sublegal']  # Retrieve the trained square root model
sqrt_pred_sublegal = randfor_sqrt_sublegal.predict(pred_X) #make predictions with obs_X features
sqrt_sublegal = np.array(sqrt_pred_sublegal) # Convert predictions to numpy array
pred['pred_sublegal'] = sqrt_sublegal**2 #reverse the square root transformation


#SPAT
randfor_sqrt_spat = trained_models_spat['randfor_sqrt_spat']  # Retrieve the trained square root model
sqrt_pred_spat = randfor_sqrt_spat.predict(pred_X) #make predictions with obs_X features
sqrt_spat = np.array(sqrt_pred_spat) # Convert predictions to numpy array
pred['pred_spat'] = sqrt_spat**2 #reverse the square root transformation


pred.to_csv('RF_predictions_2025.csv', index=False)